# Read the live `gps-raw` stream with Spark Connect

Attach to the project's Spark Connect server and consume newly arriving records from Redpanda. Spark executes the Kafka read on the server inside Docker, so it uses `redpanda-1:29092`; the notebook itself connects through the published `localhost:15002` port.

The local Jupyter kernel needs the matching Spark Connect client. In a terminal for that kernel's environment, run `pip install "pyspark[connect]==4.0.1"`, then restart the kernel. To view console-sink output, run `docker compose logs -f sedona-stream` from the project directory.

`latest` reads records arriving after the streaming query starts. Change `STARTING_OFFSETS` to `earliest` if you want to read retained history instead.

## 1. Connect to Spark

In [1]:
import os
from pyspark.sql import SparkSession

SPARK_CONNECT_URL = os.getenv("SPARK_CONNECT_URL", "sc://localhost:15002")

spark = SparkSession.builder.remote(SPARK_CONNECT_URL).getOrCreate()
print(f"Connected to Spark {spark.version} at {SPARK_CONNECT_URL}")

Connected to Spark 4.0.1 at sc://localhost:15002


## 2. Read `gps-raw` as a stream

`latest` starts with records arriving after the streaming query starts. Use `earliest` to include retained topic history.

In [4]:
KAFKA_BOOTSTRAP_SERVERS = "redpanda-1:29092"
KAFKA_TOPIC = "gps-raw"
STARTING_OFFSETS = "latest"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .load()
)

print(f"Streaming from topic {KAFKA_TOPIC} at {KAFKA_BOOTSTRAP_SERVERS}")

Streaming from topic gps-raw at redpanda-1:29092


## 3. Decode and preview messages

The Kafka `value` is binary JSON from the generator. Decode it as text without imposing a schema.

In [6]:
from pyspark.sql import functions as F

messages = raw_stream.select(
    F.col("timestamp").alias("kafka_timestamp"),
    F.col("value").cast("string").alias("payload"),
)

messages.printSchema()

root
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- payload: string (nullable = true)



## 4. Start and stop the console query

The console sink writes each micro-batch to the Spark driver logs in the `sedona-stream` container. Keep this cell running, then stop the query in the next cell when finished.

In [7]:
query = (
    messages.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)

print(f"Started streaming query: {query.id}")

Started streaming query: 77f1f30e-6ef0-46af-903b-6adbb95cbc86


In [8]:
query.stop()
print("Streaming query stopped")

Streaming query stopped
